# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB (built in Part 01) as a tool of your Agent.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

> Run `Udaplay_01` first: this notebook reads the persistent `chromadb` collection it creates.

### Setup

In [43]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [44]:
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")  # silence Chroma telemetry warnings

import ast
import contextlib
import io
import json
import re
from typing import Any, Dict, List, Optional

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.documents import Document, Corpus
from lib.llm import LLM
from lib.long_term_memory import LongTermMemory, MemoryFragment
from lib.tooling import tool
from lib.vector_db import VectorStore, VectorStoreManager

In [45]:
# Load keys from config.env (course convention) or .env, searching parent folders too.
for env_file in ("config.env", ".env"):
    env_path = find_dotenv(env_file, usecwd=True)
    if env_path:
        load_dotenv(env_path, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY", OPENAI_API_KEY)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")
os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL  # picked up by the OpenAI client used in lib/llm.py

assert OPENAI_API_KEY, "OPENAI_API_KEY is not set"
assert CHROMA_OPENAI_API_KEY, "CHROMA_OPENAI_API_KEY is not set"
assert TAVILY_API_KEY, "TAVILY_API_KEY is not set"

print("Environment variables loaded successfully.")
print("OpenAI base URL:", OPENAI_BASE_URL)

Environment variables loaded successfully.
OpenAI base URL: https://openai.vocareum.com/v1


### Vector store manager (same reusable classes as Part 01)

In [46]:
class GameVectorStore(VectorStore):
    """lib.vector_db.VectorStore plus the helpers UdaPlay needs: count / upsert / search."""

    def count(self) -> int:
        return self._collection.count()

    def upsert(self, documents: List[Document]) -> None:
        """Insert or update documents by id, so re-running the notebook never duplicates data."""
        data = Corpus(documents).to_dict()
        self._collection.upsert(
            ids=data["ids"], documents=data["contents"], metadatas=data["metadatas"]
        )

    def search(self, query: str, n_results: int = 5,
               where: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        """Semantic search returning plain dicts: game metadata + Chroma distance (lower = closer)."""
        result = self.query(query_texts=[query], n_results=n_results, where=where)
        return [
            {**metadata, "distance": round(float(distance), 4)}
            for metadata, distance in zip(result["metadatas"][0], result["distances"][0])
        ]


class PersistentVectorStoreManager(VectorStoreManager):
    """Reusable, *persistent* vector store manager.

    lib.vector_db.VectorStoreManager uses chromadb.Client() (in-memory, lost when the kernel
    stops) and the default embedding model. This subclass keeps the same interface
    (get_store / get_or_create_store / delete_store) but writes to disk and lets us choose
    the embedding model + base URL. Part 01 and Part 02 must use the same embedding model.
    """

    def __init__(self, openai_api_key: str, persist_path: str = "chromadb",
                 embedding_model: str = "text-embedding-3-small", api_base: Optional[str] = None):
        # super().__init__ is intentionally not called: it would create an in-memory client.
        self.chroma_client = chromadb.PersistentClient(path=persist_path)
        self.embedding_function = embedding_functions.OpenAIEmbeddingFunction(
            api_key=openai_api_key, model_name=embedding_model, api_base=api_base
        )

    def get_store(self, name: str) -> Optional[GameVectorStore]:
        try:
            collection = self.chroma_client.get_collection(
                name, embedding_function=self.embedding_function
            )
            return GameVectorStore(collection)
        except Exception:
            return None

    def get_or_create_store(self, store_name: str) -> GameVectorStore:
        collection = self.chroma_client.get_or_create_collection(
            name=store_name, embedding_function=self.embedding_function
        )
        return GameVectorStore(collection)

In [47]:
manager = PersistentVectorStoreManager(
    openai_api_key=CHROMA_OPENAI_API_KEY,
    persist_path="chromadb",
    embedding_model="text-embedding-3-small",  # must match Part 01
    api_base=OPENAI_BASE_URL,
)

games_store = manager.get_store("udaplay")
if games_store is None or games_store.count() == 0:
    raise RuntimeError("The 'udaplay' collection is missing or empty - run Udaplay_01 first.")
print(f"Connected to the 'udaplay' collection with {games_store.count()} games.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Connected to the 'udaplay' collection with 15 games.


### Tools

The three required tools: `retrieve_game`, `evaluate_retrieval`, `game_web_search`. Long-term memory is built into `game_web_search`.

#### Retrieve Game Tool

In [48]:
@tool
def retrieve_game(query: str) -> List[Dict[str, Any]]:
    """Semantic search: Finds the most relevant game results in the internal vector DB.

    Args:
        query: A question about the video game industry (self-contained, no pronouns).

    Returns:
        A list of the 5 closest games. Each element contains Name, Platform, Genre, Publisher,
        YearOfRelease, Description, the Chroma distance (lower = closer) and the source.
    """
    matches = games_store.search(query, n_results=5)
    for match in matches:
        match["source"] = "UdaPlay local ChromaDB"
    return matches


print(json.dumps(retrieve_game("When was Pokémon Gold and Silver released?")[:2], indent=2, ensure_ascii=False))

[
  {
    "Platform": "Game Boy Color",
    "Name": "Pokémon Gold and Silver",
    "Genre": "Role-playing",
    "YearOfRelease": 1999,
    "Publisher": "Nintendo",
    "Description": "Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.",
    "distance": 0.6528,
    "source": "UdaPlay local ChromaDB"
  },
  {
    "Platform": "Game Boy Advance",
    "Genre": "Role-playing",
    "Name": "Pokémon Ruby and Sapphire",
    "Publisher": "Nintendo",
    "Description": "Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.",
    "YearOfRelease": 2002,
    "distance": 0.9814,
    "source": "UdaPlay local ChromaDB"
  }
]


#### Evaluate Retrieval Tool

In [49]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are sufficient to answer the question")
    description: str = Field(description="Detailed explanation of the retrieval-quality decision")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence in the evaluation, from 0 to 1")


evaluation_llm = LLM(model="gpt-4o-mini", temperature=0.0)


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Assess whether the documents retrieved from the vector DB are enough to answer the question.

    Args:
        question: The original question from the user.
        retrieved_docs: The documents returned by retrieve_game.

    Returns:
        useful (bool), description (explanation of the decision) and confidence (0-1).
    """
    prompt = f"""
Your task is to evaluate whether the retrieved documents are enough to answer the user's
video-game question accurately. Give a detailed explanation so it is possible to take an
action: accept the documents or fall back to a web search.

Question:
{question}

Retrieved documents:
{json.dumps(retrieved_docs, indent=2, ensure_ascii=False)}

Judge ONLY from the retrieved documents; do not use outside knowledge or assume facts that the
documents do not state. Mark useful=true only when the documents contain the facts needed for a
well-supported answer.
Mark useful=false when the documents are unrelated, ambiguous, or do not contain the requested
game/platform/date/publisher/detail (for example the question asks about a game or a platform
that is not in the documents). Provide a confidence from 0 to 1.
"""
    response = evaluation_llm.invoke(prompt, response_format=EvaluationReport)
    return EvaluationReport.model_validate_json(response.content).model_dump()


question = "When was Pokémon Gold and Silver released?"
print(evaluate_retrieval(question, retrieve_game(question)))

{'useful': True, 'description': "The retrieved documents contain relevant information about the release year of Pokémon Gold and Silver. Specifically, one of the documents explicitly states that Pokémon Gold and Silver was released in 1999. This directly answers the user's question about the release date of the game. The other documents, while related to different Pokémon games and other Nintendo titles, do not detract from the usefulness of the first document. Therefore, the documents are sufficient to provide a well-supported answer to the question.", 'confidence': 0.9}


#### Long-term memory

Web findings are stored in a second *persistent* collection (`udaplay_long_term_memory`) so later sessions can reuse them without another web search. Saving and recalling are done **in code** inside `game_web_search` (the earlier version relied on the LLM remembering to call a `register_memory` tool, and it often did not). Every memory keeps its source URL, duplicates are skipped, and repeated questions are matched exactly. Set `RESET_LONG_TERM_MEMORY = False` to keep memories between notebook runs; it is `True` here so the demo below always starts clean and is reproducible.

In [50]:
RESET_LONG_TERM_MEMORY = True
MEMORY_STORE_NAME = "udaplay_long_term_memory"
MEMORY_OWNER, MEMORY_NAMESPACE = "udaplay-agent", "game-research"
MEMORY_HIT_MAX_DISTANCE = 0.9   # Chroma L2 distance: lower = closer. Tune if memory hits are missed / too loose.

if RESET_LONG_TERM_MEMORY:
    manager.delete_store(MEMORY_STORE_NAME)

ltm = LongTermMemory(manager, store_name=MEMORY_STORE_NAME)


def _question_key(question: str) -> str:
    """Normalised question text, used for exact-repeat lookups ('Was X on PS5?' == 'was x on ps5')."""
    return re.sub(r"\W+", " ", question.lower()).strip()


def save_web_results_to_memory(question: str, results: List[Dict[str, Any]], max_items: int = 3) -> int:
    """Persist the top web results (with their source URL) as long-term memories. Returns how many were new."""
    key, saved = _question_key(question), 0
    for item in results[:max_items]:
        if not item.get("content"):
            continue
        already = ltm.vector_store.query(
            query_texts=[question], n_results=1,
            where={"$and": [{"owner": {"$eq": MEMORY_OWNER}}, {"source_url": {"$eq": item["url"]}},
                            {"question_key": {"$eq": key}}]},
        )
        if already["documents"][0]:
            continue                                   # this URL is already stored for this question
        fact = (f"Web research for '{question}': {item['title']} - {item['content'][:400]} "
                f"Source: {item['url']}")
        ltm.register(
            MemoryFragment(content=fact, owner=MEMORY_OWNER, namespace=MEMORY_NAMESPACE),
            metadata={"source_url": item["url"], "question_key": key},
        )
        saved += 1
    return saved


def recall_from_memory(question: str, limit: int = 3) -> List[Dict[str, Any]]:
    """1) exact repeat of an earlier question, 2) otherwise semantic match closer than MEMORY_HIT_MAX_DISTANCE."""
    exact = ltm.vector_store.query(
        query_texts=[question], n_results=limit,
        where={"$and": [{"owner": {"$eq": MEMORY_OWNER}}, {"namespace": {"$eq": MEMORY_NAMESPACE}},
                        {"question_key": {"$eq": _question_key(question)}}]},
    )
    pairs = list(zip(exact["documents"][0], exact["distances"][0]))
    if not pairs:
        similar = ltm.search(question, owner=MEMORY_OWNER, namespace=MEMORY_NAMESPACE, limit=limit)
        pairs = [(f.content, d) for f, d in zip(similar.fragments, similar.metadata.get("distances", []))
                 if d <= MEMORY_HIT_MAX_DISTANCE]
    return [{"title": "Saved long-term memory", "content": content, "distance": round(float(distance), 4),
             "source": "UdaPlay long-term memory"} for content, distance in pairs]


print("Long-term memory ready:", MEMORY_STORE_NAME)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Long-term memory ready: udaplay_long_term_memory


#### Game Web Search Tool

The tool checks long-term memory **first** (deterministically, in code, instead of hoping the LLM remembers to do it) and only calls Tavily when no close memory exists.

In [51]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def game_web_search(question: str) -> List[Dict[str, Any]]:
    """Find information missing from the internal database: checks UdaPlay's saved long-term memory first, then searches the web (Tavily) and saves what it finds to long-term memory.

    Args:
        question: A self-contained question about the video game industry.

    Returns:
        Either facts saved from earlier web research (source = 'UdaPlay long-term memory'),
        or up to 5 web results with title, url, content, score (source = 'Tavily web search').
        New web findings are saved to long-term memory automatically.
    """
    # 1) Reuse knowledge saved by earlier web searches
    hits = recall_from_memory(question)
    if hits:
        return hits

    # 2) Otherwise go to the web ...
    response = tavily_client.search(
        query=question, search_depth="advanced", max_results=5, include_answer=False,
        exclude_domains=["youtube.com", "reddit.com", "fandom.com", "quora.com"],  # skip user-generated content
    )
    results = [
        {
            "title": item.get("title"),
            "url": item.get("url"),
            "content": (item.get("content") or "")[:800],
            "score": item.get("score"),
            "source": "Tavily web search",
        }
        for item in response.get("results", [])
    ]

    # 3) ... and persist what we found (a memory-write failure must never break the answer)
    try:
        saved = save_web_results_to_memory(question, results)
    except Exception as exc:
        saved = 0
        print(f"[warning] could not save to long-term memory: {exc}")
    for item in results:
        item["saved_to_long_term_memory"] = saved > 0
    return results


print("game_web_search tool ready (long-term memory first, then Tavily).")

game_web_search tool ready (long-term memory first, then Tavily).


### Agent

The `Agent` class from `lib.agents` is a state machine: `message_prep -> llm_processor <-> tool_executor -> termination`. The LLM decides in `llm_processor` whether to call a tool (go to `tool_executor`, then back) or finish. Conversation state is kept per `session_id`.

In [52]:
agent_instructions = """You are UdaPlay, an AI Research Agent for the video game industry.

Follow this workflow for every factual game-industry question, in this order:
1. Resolve follow-ups: if the question refers to earlier turns ("it", "that game", "the first one"),
   rewrite it as a fully self-contained question before calling any tool. If the earlier turns of this
   conversation already contain the answer, answer directly and say that you reused the conversation.
2. Call retrieve_game with the question (INTERNAL knowledge always comes first).
3. Call evaluate_retrieval with the original question and ALL documents exactly as returned by
   retrieve_game. Do not filter, shorten or rewrite them: the evaluator must see everything that was retrieved.
4. If useful=true: answer using only those internal documents.
5. If useful=false: call game_web_search. It checks UdaPlay's long-term memory first and automatically
   saves new web findings to long-term memory, so never try to save anything yourself. Look at the
   'source' field of the results:
   a. 'UdaPlay long-term memory': facts saved from earlier web research. Answer from them, cite
      'UdaPlay long-term memory' and the original URL that appears at the end of each saved text.
   b. 'Tavily web search': answer from the web evidence (you may combine it with relevant internal
      documents). Prefer official or major-outlet sources, mention conflicts between sources, and
      do not rely on forum posts.

Rules:
- Never state a platform, release date, developer, publisher or current project unless it is supported
  by tool evidence. If the evidence is thin or contradictory, say so and lower the confidence.
- Cite every source your answer relies on, and keep the stated confidence consistent with them.
- If a claim such as 'first', 'latest' or 'only' is inferred from the dataset (for example from release
  years) rather than stated in a document or source, say that it is inferred.
- Be concise and write for a business reader.

Final answer format (markdown):
**Answer:** the direct answer in 1-3 sentences.
**Reasoning:** 1-2 sentences naming the tools you used and why (e.g. internal documents were / were not sufficient).
**Sources:** a bullet list using 'UdaPlay local ChromaDB', 'UdaPlay long-term memory' or '[title](url)' for web results.
**Confidence:** High / Medium / Low with a short justification.
"""


def build_agent() -> Agent:
    return Agent(
        model_name="gpt-4o-mini",
        instructions=agent_instructions,
        tools=[retrieve_game, evaluate_retrieval, game_web_search],
        temperature=0.0,
    )


udaplay_agent = build_agent()
print("UdaPlay state-machine agent ready with tools:", [t.name for t in udaplay_agent.tools])

UdaPlay state-machine agent ready with tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']


### Reporting

`ask()` runs the agent and prints a report for **that turn only**: the state-machine path, every tool call with a summary of its result (this is where the evaluator's reasoning shows), and the final cited answer.

In [53]:
def _summarize_tool_result(name: str, content: str) -> str:
    """Turn a ToolMessage payload into one readable line."""
    try:
        data = json.loads(content)              # the agent JSON-encodes str(result)
        if isinstance(data, str):
            try:
                data = ast.literal_eval(data)
            except (ValueError, SyntaxError):
                return data[:300]
        if name == "retrieve_game":
            return "; ".join(f"{d['Name']} [{d['Platform']}, {d['YearOfRelease']}] dist={d['distance']}" for d in data)
        if name == "evaluate_retrieval":
            return f"useful={data['useful']} (confidence {data['confidence']}) - {data['description']}"
        if name == "game_web_search":
            if data and data[0].get("source") == "UdaPlay long-term memory":
                return "LONG-TERM MEMORY HIT (no web call): " + "; ".join(
                    f"{d['content'][:160]} (dist={d['distance']})" for d in data)
            saved = " [saved to long-term memory]" if data and data[0].get("saved_to_long_term_memory") else ""
            return "WEB: " + "; ".join(f"{d['title']} <{d['url']}>" for d in data) + saved
        return str(data)[:300]
    except Exception:
        return str(content)[:300]


def _format_args(arguments: str) -> str:
    args = json.loads(arguments)
    return ", ".join(
        f"{k}=<{len(v)} items>" if isinstance(v, list) else f"{k}={str(v)[:140]!r}"
        for k, v in args.items()
    )


def print_report(question: str, run) -> None:
    state = run.get_final_state()
    messages = state["messages"]
    last_user = max(i for i, m in enumerate(messages) if m.role == "user")
    turn = messages[last_user + 1:]

    print("=" * 100)
    print("QUESTION:", question)
    print("STATE MACHINE PATH:", " -> ".join(s.step_id for s in run.snapshots))
    print("TOOL USAGE:")
    tool_calls = 0
    for m in turn:
        if m.role == "assistant" and m.tool_calls:
            for call in m.tool_calls:
                tool_calls += 1
                print(f"  [{tool_calls}] {call.function.name}({_format_args(call.function.arguments)})")
        elif m.role == "tool":
            print(f"      -> {_summarize_tool_result(m.name, m.content)}")
    if not tool_calls:
        print("  (no tools - answered from the conversation context)")
    final = [m for m in turn if m.role == "assistant" and m.content][-1].content
    print(f"\nFINAL ANSWER:\n{final}")
    print(f"\n(tokens used this turn: {state.get('total_tokens', 'n/a')})")


def ask(agent: Agent, question: str, session_id: str):
    # The state machine logs every step to stdout; hide that noise, the report shows the path instead.
    with contextlib.redirect_stdout(io.StringIO()):
        run = agent.invoke(question, session_id=session_id)
    print_report(question, run)
    return run

### Invoke your agent

The three example queries from the activity (same session, so the agent keeps context):

In [54]:
session_id = "udaplay-demo"

example_queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for question in example_queries:
    ask(udaplay_agent, question, session_id)

QUESTION: When was Pokémon Gold and Silver released?
STATE MACHINE PATH: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor
TOOL USAGE:
  [1] retrieve_game(query='When was Pokémon Gold and Silver released?')
      -> Pokémon Gold and Silver [Game Boy Color, 1999] dist=0.6528; Pokémon Ruby and Sapphire [Game Boy Advance, 2002] dist=0.9814; Super Mario World [Super Nintendo Entertainment System (SNES), 1990] dist=1.4669; Super Mario 64 [Nintendo 64, 1996] dist=1.4733; Mario Kart 8 Deluxe [Nintendo Switch, 2017] dist=1.4979
  [2] evaluate_retrieval(question='When was Pokémon Gold and Silver released?', retrieved_docs=<5 items>)
      -> useful=True (confidence 0.9) - The retrieved documents contain relevant information about the release date of Pokémon Gold and Silver. Specifically, one of the documents explicitly states that Pokémon Gold and Silver were released in 1999. This directly answers the user's question about the releas

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


QUESTION: Was Mortal Kombat X released for PlayStation 5?
STATE MACHINE PATH: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor
TOOL USAGE:
  [1] retrieve_game(query='Was Mortal Kombat X released for PlayStation 5?')
      -> Marvel's Spider-Man 2 [PlayStation 5, 2023] dist=1.0402; Marvel's Spider-Man [PlayStation 4, 2018] dist=1.2044; Gran Turismo 5 [PlayStation 3, 2010] dist=1.2058; Halo Infinite [Xbox Series X|S, 2021] dist=1.3914; Mario Kart 8 Deluxe [Nintendo Switch, 2017] dist=1.4048
  [2] evaluate_retrieval(question='Was Mortal Kombat X released for PlayStation 5?', retrieved_docs=<5 items>)
      -> useful=False (confidence 0.9) - The retrieved documents do not contain any information regarding Mortal Kombat X or its release on PlayStation 5. Instead, they focus on other games such as Marvel's Spider-Man and Gran Turismo 5, which are unrelated to the user's question. Since there is no

#### Multi-turn memory (same session)

This follow-up only makes sense if the agent remembers the earlier turns.

In [55]:
ask(udaplay_agent,
    "Going back to the first game I asked about: who published it, and what genre is it?",
    session_id)

QUESTION: Going back to the first game I asked about: who published it, and what genre is it?
STATE MACHINE PATH: __entry__ -> message_prep -> llm_processor
TOOL USAGE:
  (no tools - answered from the conversation context)

FINAL ANSWER:
**Answer:** Pokémon Gold and Silver were published by Nintendo and belong to the role-playing genre.  
**Reasoning:** This information was retrieved from the internal documents that provided details about the game.  
**Sources:**  
- UdaPlay local ChromaDB  
**Confidence:** High, based on the explicit information retrieved.

(tokens used this turn: 5594)


Run('bca8ae92-8fd9-4403-9656-7e589b80c6a4')

### Long-term memory across sessions

A **brand-new agent** with a **new session** has an empty short-term memory. Because the Mortal Kombat X web findings were saved to long-term memory, `game_web_search` should now find them there and skip Tavily.

In [56]:
fresh_agent = build_agent()
run = ask(fresh_agent, "Was Mortal Kombat X released for PlayStation 5?", session_id="new-session")

tool_outputs = " ".join(m.content for m in run.get_final_state()["messages"] if m.role == "tool")
if "LONG-TERM MEMORY" in tool_outputs.upper() or "UdaPlay long-term memory" in tool_outputs:
    print("\nCHECK: answered from long-term memory (no Tavily call).")
else:
    print("\nCHECK: memory was NOT reused - see MEMORY_HIT_MAX_DISTANCE.")

print("\nStored long-term memories matching the topic:")
saved = ltm.search("Mortal Kombat X PlayStation 5", owner=MEMORY_OWNER, namespace=MEMORY_NAMESPACE, limit=3)
for fragment in saved.fragments:
    print("-", fragment.content)
if not saved.fragments:
    print("(none - nothing was saved; check the 'saved to long-term memory' note in the Mortal Kombat X report above)")

QUESTION: Was Mortal Kombat X released for PlayStation 5?
STATE MACHINE PATH: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor
TOOL USAGE:
  [1] retrieve_game(query='Was Mortal Kombat X released for PlayStation 5?')
      -> Marvel's Spider-Man 2 [PlayStation 5, 2023] dist=1.0402; Marvel's Spider-Man [PlayStation 4, 2018] dist=1.2044; Gran Turismo 5 [PlayStation 3, 2010] dist=1.2058; Halo Infinite [Xbox Series X|S, 2021] dist=1.3914; Mario Kart 8 Deluxe [Nintendo Switch, 2017] dist=1.4048
  [2] evaluate_retrieval(question='Was Mortal Kombat X released for PlayStation 5?', retrieved_docs=<5 items>)
      -> useful=False (confidence 0.9) - The retrieved documents do not contain any information regarding Mortal Kombat X or its availability on PlayStation 5. Instead, they focus on other games such as Marvel's Spider-Man and Gran Turismo 5, which are unrelated to the user's question. Since there 

### (Optional) Advanced

- **Long-term memory:** implemented above (memory-first lookup and automatic saving inside `game_web_search`, persistent Chroma collection).
- **State machine:** the agent already runs as a state machine (`message_prep -> llm_processor <-> tool_executor -> termination`). Turning each tool into its own fixed node would make the retrieve -> evaluate -> web-search order deterministic instead of prompt-driven; that would require changing `lib/agents.py`, which was left untouched.